# Notebook 3: Thử Nghiệm So Sánh 2 Phương Pháp Phân Đoạn U Não
**Đồ án Xử lý ảnh số (2026–2027) — Khoa CNTT — HUIT**  
**Nhóm sinh viên:** Hà Mạnh Trường (2045240326) & Huỳnh Tâm Trí (2045240315)  
**Phụ trách chính:** Cả hai thành viên

---
### Tiêu chí Bộ môn (Chiếm 15% + 50% điểm đồ án):
Bắt buộc so sánh tối thiểu **02 phương án hoặc cấu hình xử lý**:
* **Phương án A (Baseline chính):** **Marker-controlled Watershed** (Biến đổi khoảng cách Euclidean $ightarrow$ Hạt giống $ightarrow$ Phân thủy).
* **Phương án B (Đối chứng):** **Seeded Region Growing** (Tự tìm hạt giống cực đại cường độ $ightarrow$ Lan truyền 8 láng giềng).


In [ ]:
import os
import sys
import numpy as np
import cv2
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from src.preprocessing.skull_strip import strip_skull
from src.segmentation.watershed import segment_watershed
from src.segmentation.region_growing import segment_region_growing
from src.measurement.features import extract_tumor_features
from src.measurement.visualization import draw_tumor_overlay
from src.evaluation.metrics import calculate_dice, calculate_iou


### 1. So Sánh Trực Quan Trên 3 Ca Điển Hình (U Nhỏ, U Trung Bình, U Lớn)


In [ ]:
cases = [
    ("mri_brain_tumor_11", "Ca U Nhỏ (1.12%)"),
    ("mri_brain_tumor_02", "Ca U Trung Bình (4.04%)"),
    ("mri_brain_tumor_07", "Ca U Lớn (9.44%)")
]

fig, axes = plt.subplots(3, 5, figsize=(18, 11))

for row_idx, (case_id, desc) in enumerate(cases):
    img_p = f"../data/sample/images/{case_id}.png"
    mask_p = f"../data/sample/masks/{case_id}_mask.png"
    
    orig = cv2.imread(img_p, cv2.IMREAD_GRAYSCALE)
    gt = cv2.imread(mask_p, cv2.IMREAD_GRAYSCALE)
    stripped, b_mask = strip_skull(orig)
    
    # 1. Chạy Watershed
    mask_ws = segment_watershed(stripped, dist_thresh_ratio=0.4)
    dice_ws = calculate_dice(mask_ws, gt)
    
    # 2. Chạy Region Growing
    mask_rg = segment_region_growing(stripped, tolerance=15)
    dice_rg = calculate_dice(mask_rg, gt)
    
    # 3. Vẽ Bounding Box cảnh báo
    res_ws = extract_tumor_features(mask_ws, b_mask)
    annotated = draw_tumor_overlay(orig, mask_ws, res_ws)
    
    # Cột 1: Ảnh gốc
    axes[row_idx, 0].imshow(orig, cmap='gray')
    axes[row_idx, 0].set_title(f"{desc}\nẢnh Gốc", fontsize=10)
    axes[row_idx, 0].axis('off')
    
    # Cột 2: Ground Truth
    axes[row_idx, 1].imshow(gt, cmap='gray')
    axes[row_idx, 1].set_title("Ground Truth Bác Sĩ", fontsize=10)
    axes[row_idx, 1].axis('off')
    
    # Cột 3: Watershed
    axes[row_idx, 2].imshow(mask_ws, cmap='Blues')
    axes[row_idx, 2].set_title(f"Watershed\nDice = {dice_ws:.4f}", fontsize=10, color='blue', fontweight='bold')
    axes[row_idx, 2].axis('off')
    
    # Cột 4: Region Growing
    axes[row_idx, 3].imshow(mask_rg, cmap='Greens')
    axes[row_idx, 3].set_title(f"Region Growing\nDice = {dice_rg:.4f}", fontsize=10, color='green', fontweight='bold')
    axes[row_idx, 3].axis('off')
    
    # Cột 5: Bounding Box
    axes[row_idx, 4].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    axes[row_idx, 4].set_title(f"Khoanh Vùng Đỏ\nArea: {res_ws.tumor_area_px}px", fontsize=10)
    axes[row_idx, 4].axis('off')

plt.suptitle("So Sánh Hiệu Quả Phân Đoạn: Watershed (Baseline) vs. Region Growing (Đối Chứng)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
